# Gradient descent from scratch

In [ ]:
# Iimport all the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# load the housing dataset
df = pd.read_csv(
    "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
)

In [ ]:
df.head()

## Step 1 — setup, leakage-safe scaling

Single feature to start: `median_income` → `median_house_value`.

Scaling is required here (not just good practice) — `median_income` ranges ~0-15 while the target is in the hundreds of thousands. Feeding that scale mismatch into gradient descent with a normal learning rate causes slow/bad convergence or outright divergence.

We fit the scaler (mean/std) on the **train** split only, then apply those same numbers to the test split — same leakage discipline as every other day this week.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[["median_income"]].values
y = df["median_house_value"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
# scale using TRAIN stats only — same leakage discipline as every day this week
x_mean, x_std = X_train.mean(), X_train.std()

X_train_s = (X_train - x_mean) / x_std
X_test_s = (X_test - x_mean) / x_std

print("x_mean:", x_mean, "x_std:", x_std)
print("scaled train mean/std:", X_train_s.mean(), X_train_s.std())

## Step 2 — cost function and gradient, by hand

`cost_fn` = mean squared error (MSE) — how wrong the line is.
`gradients` = derivative of MSE w.r.t. `w` and `b` — which direction (and how much) to nudge each.

In [ ]:
def cost_fn(w: float, b: float, X: np.ndarray, y: np.ndarray) -> float:
    # model's predictions for every house: pred = w*x + b
    preds = w * X.flatten() + b
    # mean squared error: average of (prediction - actual)^2 across all houses
    return np.mean((preds - y) ** 2)


def gradients(w: float, b: float, X: np.ndarray, y: np.ndarray) -> tuple[float, float]:
    n = len(y)
    preds = w * X.flatten() + b
    error = (
        preds - y
    )  # how far off each prediction is (+ve = overshot, -ve = undershot)

    # dw: derivative of MSE w.r.t. w -> scaled by x, since w's effect on the
    # prediction depends on x (pred = w*x + b)
    dw = (2 / n) * np.sum(error * X.flatten())

    # db: derivative of MSE w.r.t. b -> not scaled by x, since b shifts every
    # prediction by the same amount regardless of x
    db = (2 / n) * np.sum(error)

    return dw, db

## Step 3 — the update loop

Repeatedly compute the gradient and take a small step downhill (opposite to the gradient), tracking cost so we can see it converge.

In [ ]:
# start from a neutral guess: a flat line predicting 0 for every house
w, b = 0.0, 0.0

lr = 0.1  # step size: how far to move along the gradient direction each iteration
cost_history = []  # track cost each iteration so we can see it converge

for i in range(1000):
    dw, db = gradients(w, b, X_train_s, y_train)

    # subtract because the gradient points uphill (toward higher cost) —
    # we want to move downhill (toward lower cost)
    w -= lr * dw
    b -= lr * db

    cost_history.append(cost_fn(w, b, X_train_s, y_train))

print("final w:", w, "final b:", b)
print("final cost:", cost_history[-1])

## Step 4 — compare to sklearn

sklearn solves for w/b algebraically (no gradient descent) — if our iterative answer matches, that's strong evidence the from-scratch math is correct.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# fit on the identical scaled data our from-scratch loop used
sk_model = LinearRegression().fit(X_train_s, y_train)

# sklearn's w is coef_[0] (one feature), b is intercept_
print("from-scratch: w =", w, "b =", b)
print("sklearn:      w =", sk_model.coef_[0], "b =", sk_model.intercept_)

# compare predictive error on the held-out test set, not just w/b
from_scratch_test_preds = w * X_test_s.flatten() + b
sk_test_preds = sk_model.predict(X_test_s)

print("from-scratch test MSE:", mean_squared_error(y_test, from_scratch_test_preds))
print("sklearn test MSE:     ", mean_squared_error(y_test, sk_test_preds))

## Step 5 — gradient check

Verify the hand-derived `dw`/`db` formulas against a numerical (finite-difference) approximation — independent of whether the final trained answer happened to match sklearn.

In [ ]:
eps = 1e-4  # tiny nudge for the finite-difference approximation
w_test, b_test = 0.5, 1.0  # arbitrary point to check the gradient at

# analytical: straight from our hand-derived formula
dw_analytical, db_analytical = gradients(w_test, b_test, X_train_s, y_train)

# numerical: measure the slope directly by nudging w slightly up and down
# and seeing how much the cost changes — no derivative formula involved
dw_numerical = (
    cost_fn(w_test + eps, b_test, X_train_s, y_train)
    - cost_fn(w_test - eps, b_test, X_train_s, y_train)
) / (2 * eps)

# same idea, nudging b instead
db_numerical = (
    cost_fn(w_test, b_test + eps, X_train_s, y_train)
    - cost_fn(w_test, b_test - eps, X_train_s, y_train)
) / (2 * eps)

print("dw: analytical =", dw_analytical, " numerical =", dw_numerical)
print("db: analytical =", db_analytical, " numerical =", db_numerical)

# should agree to ~5 significant figures; remaining gap is just the
# finite-difference approximation itself, not a bug
